# Can we forecast next month's risk?

**Question.** Volatility, meaning how much prices swing, matters for anyone deciding how much risk to take. It is also much more predictable than the direction of prices. How well can simple models forecast **next month's volatility**, and does a proper statistical model (GARCH) beat simple rules of thumb?

**Setup**
- **Target:** realised volatility over the next 21 trading days (about a month), annualised.
- **Walk-forward evaluation:** each forecast made on day *t* uses only data up to day *t*. Models are refitted every 21 days on all data available at that point. The first 500 days are used only for training. A unit test (`test_walk_forward_has_no_look_ahead`) rewrites future prices and checks that past forecasts do not change.
- **Metrics:** RMSE (error in volatility points) and **QLIKE**, the standard loss for variance forecasts. QLIKE punishes forecasts that are too low more than ones that are too high, which matches how risk forecasts are used.

| Model | Idea |
|---|---|
| `last_month` | next month looks like the last month (baseline) |
| `long_run` | the average volatility so far |
| `ewma` | RiskMetrics: recent days weighted more (lambda 0.94) |
| `garch` | GARCH(1,1) by maximum likelihood: shocks raise volatility, which then decays back to a long-run level |
| `har` | HAR regression: next month's variance from the last day, week and month |

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "backend"))
from app import analytics as an
from app import backtest as bt
from app import forecast as fc
from app.config import WATCHLIST_GROUPS

NAMES = {s: n for g in WATCHLIST_GROUPS.values() for s, n in g.items()}
import datavest_style  # site colours and fonts
pd.options.display.float_format = "{:,.3f}".format

with sqlite3.connect(Path("..") / "backend" / "datavest.db") as con:
    long = pd.read_sql("SELECT t.symbol, p.date, p.adj_close, p.close FROM prices p JOIN tickers t ON t.id = p.ticker_id",
                       con, parse_dates=["date"])
long["adj_close"] = long["adj_close"].fillna(long["close"])
prices = long.pivot(index="date", columns="symbol", values="adj_close").sort_index()
prices = prices[[s for s in NAMES if s in prices.columns]]
print(f"{prices.shape[1]} markets, {prices.index[0].date()} to {prices.index[-1].date()}")

## 1. Volatility clusters

Calm and stormy periods come in runs. That is what makes volatility forecastable at all.

In [ ]:
r = fc.log_returns(prices["^HSI"].dropna())
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(r.index, r * 100, lw=0.6, color="#1b2433"); axes[0].set_ylabel("daily return %")
axes[0].set_title("Hang Seng: daily returns")
lags = range(1, 41)
axes[1].bar(list(lags), [r.abs().autocorr(l) for l in lags], color="#2a78d6", label="|returns|")
axes[1].bar(list(lags), [r.autocorr(l) for l in lags], color="#eb6834", alpha=0.8, label="returns")
axes[1].set_title("Autocorrelation by lag (days): returns ≈ 0, size of returns clearly > 0")
axes[1].set_xlabel("lag (trading days)"); axes[1].legend()
fig.tight_layout()

Returns themselves have almost no memory, which is why direction is hard to predict. The **size** of returns does: a volatile day is usually followed by more volatile days.

## 2. Walk-forward forecasts for every market

In [ ]:
results, tables = {}, {}
for sym in prices:
    close = prices[sym].dropna()
    results[sym] = fc.walk_forward(close, h=21, min_train=500, refit_every=21)
    tables[sym] = fc.evaluate(results[sym])
n = {s: len(results[s]) for s in results}
print("out-of-sample forecasts per market:", min(n.values()), "to", max(n.values()))

In [ ]:
qlike = pd.DataFrame({NAMES[s]: t["qlike"] for s, t in tables.items()}).T
rel = qlike.div(qlike["last_month"], axis=0) - 1
summary = pd.DataFrame({
    "avg QLIKE vs last_month": rel.mean(),
    "markets where it beats last_month": (qlike.lt(qlike["last_month"], axis=0)).sum(),
    "markets where it is the best model": qlike.idxmin(axis=1).value_counts().reindex(qlike.columns).fillna(0).astype(int),
}).loc[fc.MODELS].rename_axis(None)
summary.style.format({"avg QLIKE vs last_month": "{:+.0%}"})

In [ ]:
rel.style.format("{:+.0%}").background_gradient(cmap="RdYlGn_r", vmin=-0.4, vmax=0.4).set_caption(
    "QLIKE error relative to the last-month baseline (negative = better)")

## 3. What the forecasts look like

In [ ]:
sym = "^HSI"
f = results[sym]
ax = f["target"].plot(color="#1b2433", lw=1.2, label="what actually happened (next 21 days)")
f["garch"].plot(ax=ax, color="#2a78d6", lw=1.2, label="GARCH forecast")
f["last_month"].plot(ax=ax, color="#eb6834", lw=1, alpha=0.8, label="last month (baseline)")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}"); ax.set_title(f"{NAMES[sym]}: forecast vs realised volatility"); ax.legend(); ax.set_xlabel("");

Note how the baseline **lags**: it only reacts after a calm or stormy month is over. GARCH reacts within days of a shock and also expects a spike to fade.

## 4. Inside the GARCH model

Fitted on all available data. *Persistence* (alpha + beta) says how slowly a shock fades, and the *half-life* is how many days it takes to fade halfway.

In [ ]:
rows = {}
for sym in prices:
    r = fc.log_returns(prices[sym].dropna()) * 100
    g = fc.garch_fit(r.to_numpy())
    rows[NAMES[sym]] = {"alpha (reaction)": g.alpha, "beta (memory)": g.beta, "persistence": g.persistence,
                        "half-life (days)": np.log(0.5) / np.log(g.persistence),
                        "long-run volatility": np.sqrt(g.long_run_var * 252) / 100}
pd.DataFrame(rows).T.style.format({"alpha (reaction)": "{:.3f}", "beta (memory)": "{:.3f}", "persistence": "{:.3f}",
                                   "half-life (days)": "{:.0f}", "long-run volatility": "{:.0%}"})

## 5. Findings

In [ ]:
best = summary["avg QLIKE vs last_month"].idxmin()
lines = [
    f"Across {len(tables)} markets, the best model on average was **{best}**, with QLIKE {summary.loc[best, 'avg QLIKE vs last_month']:+.0%} versus the last-month baseline.",
    f"GARCH beat the baseline in {summary.loc['garch', 'markets where it beats last_month']} of {len(tables)} markets; EWMA in {summary.loc['ewma', 'markets where it beats last_month']}; HAR in {summary.loc['har', 'markets where it beats last_month']}.",
    f"The long-run average was {'worse' if summary.loc['long_run', 'avg QLIKE vs last_month'] > 0 else 'better'} than the baseline on average ({summary.loc['long_run', 'avg QLIKE vs last_month']:+.0%}): volatility regimes matter.",
]
from IPython.display import Markdown
Markdown("\n\n".join("- " + l for l in lines))

## Limitations

- **Short test window.** With 5 years of data and 500 days for training, each market has about 3 years of out-of-sample forecasts. Rankings between close models (GARCH, EWMA, HAR) can flip with a different period. On simulated data with 3,000 days, GARCH wins consistently (see the unit tests).
- **Overlapping targets.** Daily forecasts of 21-day windows overlap, so errors are correlated and the effective sample is smaller than the row count. A formal comparison would use a Diebold–Mariano test with HAC standard errors.
- **Gaussian GARCH.** Real returns have fatter tails. A Student-t GARCH or GJR-GARCH (which reacts more to falls than rises) is a natural next step.
- **Forecasting volatility is not forecasting direction.** A good risk forecast tells you how bumpy the ride may be, not whether prices will go up.